<a href="https://colab.research.google.com/github/razabpn/Hello-World/blob/main/Whatsapp_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas nltk textblob scikit-learn
import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [5]:
import pandas as pd
import re

# Read the chat file
# Assuming the file is named 'chat.txt' based on your previous example
with open('/content/WhatsApp Chat with Mayuri Deshmukh.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Parse lines into a list
data = []
# Initialize variables to handle multi-line messages
current_timestamp = None
current_sender = None
current_message = ""

# Regex to match the start of a new message line
message_start_pattern = re.compile(r'(\d{2}/\d{2}/\d{4}, \d{2}:\d{2}) - (.+?): (.+)')
# Regex to match the initial "Messages and calls are end-to-end encrypted" line
encryption_message_pattern = re.compile(r'(\d{2}/\d{2}/\d{4}, \d{2}:\d{2}) - Messages and calls are end-to-end encrypted.*')

for line in lines:
    line = line.strip() # Remove leading/trailing whitespace
    if not line: # Skip empty lines
        continue

    # Check if the line is the encryption message
    encryption_match = encryption_message_pattern.match(line)
    if encryption_match:
        # If there was a previous message being built, add it to data
        if current_timestamp and current_sender and current_message:
            data.append([current_timestamp, current_sender, current_message.strip()])
        # Reset for the new "encryption" entry
        timestamp = encryption_match.groups()[0]
        sender = "System" # Assign a sender for system messages
        message = line[len(encryption_match.group(0)) - len(line):] # Capture the whole line as message
        data.append([timestamp, sender, message])
        # Reset multi-line tracking
        current_timestamp = None
        current_sender = None
        current_message = ""
        continue

    # Check if the line is the start of a new regular message
    match = message_start_pattern.match(line)
    if match:
        # If there was a previous message being built, add it to data
        if current_timestamp and current_sender and current_message:
            data.append([current_timestamp, current_sender, current_message.strip()])

        # Start a new message
        current_timestamp, current_sender, first_line_message = match.groups()
        current_message = first_line_message + "\n" # Add the first line to the message
    else:
        # If the line is not the start of a new message, it's part of the previous message
        # Only append if we are currently tracking a message
        if current_timestamp is not None:
            current_message += line + "\n"

# Add the last message to data if it exists
if current_timestamp and current_sender and current_message:
    data.append([current_timestamp, current_sender, current_message.strip()])


# Create a DataFrame
df = pd.DataFrame(data, columns=['Timestamp', 'Sender', 'Message'])
print(df.head())  # Show first 5 rows

           Timestamp           Sender  \
0  02/03/2022, 14:49           System   
1  02/03/2022, 14:49  Mayuri Deshmukh   
2  02/03/2022, 14:49  Mayuri Deshmukh   
3  02/03/2022, 14:49            Biswa   
4  03/03/2022, 10:53  Mayuri Deshmukh   

                                             Message  
0  02/03/2022, 14:49 - Messages and calls are end...  
1                                                 Hi  
2                                        Mayuri here  
3                                    <Media omitted>  
4                                                 Hi  


In [6]:
from textblob import TextBlob
import pandas as pd

# Define romantic keywords
romantic_keywords = ['love', 'miss you', 'babe', 'darling', 'honey', 'date', 'boyfriend', 'girlfriend']

# Analyze sentiment and keywords
def analyze_message(message):
    # Sentiment analysis
    blob = TextBlob(message)
    sentiment = blob.sentiment.polarity  # -1 (negative) to 1 (positive)
    # Count romantic keywords
    keyword_count = sum(1 for keyword in romantic_keywords if keyword.lower() in message.lower())
    return sentiment, keyword_count

# Apply analysis to each message
df['Sentiment'], df['Keyword_Count'] = zip(*df['Message'].apply(analyze_message))

# Summarize by sender
summary = df.groupby('Sender').agg({
    'Message': 'count',
    'Sentiment': 'mean',
    'Keyword_Count': 'sum'
}).rename(columns={'Message': 'Message_Count'})
print(summary)

                 Message_Count  Sentiment  Keyword_Count
Sender                                                  
Biswa                      531   0.020321              0
Mayuri Deshmukh            717   0.015514              0
System                       1   0.250000              0


In [7]:
# Simple prediction logic
def predict_relationship(summary):
    # If any contact has high message count and romantic keywords
    for sender, row in summary.iterrows():
        if row['Message_Count'] > 20 and row['Keyword_Count'] > 5 and row['Sentiment'] > 0.2:
            return f"In a Relationship (likely with {sender})"
    return "Single or unclear"

# Run prediction
prediction = predict_relationship(summary)
print("Predicted Status:", prediction)
print("\nKey Indicators:")
print(summary)

Predicted Status: Single or unclear

Key Indicators:
                 Message_Count  Sentiment  Keyword_Count
Sender                                                  
Biswa                      531   0.020321              0
Mayuri Deshmukh            717   0.015514              0
System                       1   0.250000              0


In [8]:
print(df.head())
print(df['Sender'].value_counts())

           Timestamp           Sender  \
0  02/03/2022, 14:49           System   
1  02/03/2022, 14:49  Mayuri Deshmukh   
2  02/03/2022, 14:49  Mayuri Deshmukh   
3  02/03/2022, 14:49            Biswa   
4  03/03/2022, 10:53  Mayuri Deshmukh   

                                             Message  Sentiment  Keyword_Count  
0  02/03/2022, 14:49 - Messages and calls are end...       0.25              0  
1                                                 Hi       0.00              0  
2                                        Mayuri here       0.00              0  
3                                    <Media omitted>       0.00              0  
4                                                 Hi       0.00              0  
Sender
Mayuri Deshmukh    717
Biswa              531
System               1
Name: count, dtype: int64


In [9]:
df['Word_Count'] = df['Message'].apply(lambda x: len(x.split()))
print(df.groupby('Sender')['Word_Count'].mean())

Sender
Biswa               4.111111
Mayuri Deshmukh     2.294282
System             28.000000
Name: Word_Count, dtype: float64


In [10]:
!pip install vaderSentiment


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.5 MB/s eta 0:00:00


In [11]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [12]:
# Simplified LIWC-like dictionary
personality_words = {
    'Openness': ['new', 'idea', 'creative', 'explore', 'art', 'imagine'],
    'Conscientiousness': ['plan', 'work', 'done', 'organize', 'task', 'goal'],
    'Extraversion': ['party', 'fun', 'talk', 'meet', 'energy', 'excited'],
    'Agreeableness': ['help', 'care', 'sorry', 'thanks', 'kind', 'support'],
    'Neuroticism': ['worry', 'stress', 'sad', 'afraid', 'nervous', 'upset']
}

In [13]:
# Initialize VADER
analyzer = SentimentIntensityAnalyzer()

# Analyze messages
def analyze_message(message):
    # Sentiment (compound score: -1 to 1)
    sentiment = analyzer.polarity_scores(message)['compound']
    # Count personality words
    word_counts = {trait: 0 for trait in personality_words}
    for trait, words in personality_words.items():
        word_counts[trait] = sum(1 for word in words if word.lower() in message.lower())
    return sentiment, word_counts

# Apply analysis
df['Sentiment'] = df['Message'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
personality_scores = df['Message'].apply(analyze_message)
df['Openness'] = [scores[1]['Openness'] for scores in personality_scores]
df['Conscientiousness'] = [scores[1]['Conscientiousness'] for scores in personality_scores]
df['Extraversion'] = [scores[1]['Extraversion'] for scores in personality_scores]
df['Agreeableness'] = [scores[1]['Agreeableness'] for scores in personality_scores]
df['Neuroticism'] = [scores[1]['Neuroticism'] for scores in personality_scores]

# Summarize by sender
summary = df.groupby('Sender').agg({
    'Message': 'count',
    'Sentiment': 'mean',
    'Word_Count': 'mean',
    'Openness': 'sum',
    'Conscientiousness': 'sum',
    'Extraversion': 'sum',
    'Agreeableness': 'sum',
    'Neuroticism': 'sum'
}).rename(columns={'Message': 'Message_Count'})
print(summary)

                 Message_Count  Sentiment  Word_Count  Openness  \
Sender                                                            
Biswa                      531   0.071081    4.111111         7   
Mayuri Deshmukh            717   0.096273    2.294282         2   
System                       1  -0.296000   28.000000         0   

                 Conscientiousness  Extraversion  Agreeableness  Neuroticism  
Sender                                                                        
Biswa                            2             2              2            1  
Mayuri Deshmukh                  0             1              0            0  
System                           0             0              0            0  


In [14]:
# Normalize scores (0-1) based on max word count
def normalize_scores(summary):
    traits = ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']
    for trait in traits:
        max_score = summary[trait].max()
        if max_score > 0:
            summary[trait] = summary[trait] / max_score
    return summary

# Add heuristic adjustments
def predict_personality(summary):
    summary = normalize_scores(summary)
    # Adjust based on message count (Extraversion) and sentiment (Neuroticism)
    summary['Extraversion'] = summary['Extraversion'] * (summary['Message_Count'] / summary['Message_Count'].max())
    summary['Neuroticism'] = summary['Neuroticism'] * (1 - summary['Sentiment'])  # Lower sentiment -> higher Neuroticism
    return summary

# Run prediction
personality_summary = predict_personality(summary)
print("Big Five Personality Predictions (0-1):")
print(personality_summary[['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']])

Big Five Personality Predictions (0-1):
                 Openness  Conscientiousness  Extraversion  Agreeableness  \
Sender                                                                      
Biswa            1.000000                1.0      0.740586            1.0   
Mayuri Deshmukh  0.285714                0.0      0.500000            0.0   
System           0.000000                0.0      0.000000            0.0   

                 Neuroticism  
Sender                        
Biswa               0.928919  
Mayuri Deshmukh     0.000000  
System              0.000000  
